In [ ]:
import pandas as pd
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import matplotlib.dates as mdates
import seaborn as sns
from matplotlib.collections import LineCollection
import glob
import re
from datetime import timedelta
import os
from collections import defaultdict
from scipy import stats
import math
from pathlib import Path

# MEDA A e MEDA B

## Uniformate datetime

In [ ]:
def uniforma_date(df, datetime_col='Datetime'):
    """
    Standardizes the dates in the dataset by converting them all to DD/MM/YYYY HH:MM:SS format
    without adding rows for missing dates.

    Parameters:
    - df: DataFrame containing the data
    - datetime_col: name of the datetime column
    """
    original_len = len(df)
    df = df.dropna(subset=[datetime_col]).reset_index(drop=True)
    df[datetime_col] = df[datetime_col].astype(str)

    # --- delete microsec/nanosec ---
    cleaned_count = 0
    for idx in range(len(df)):
        timestamp_str = df.at[idx, datetime_col]
        if '.' in timestamp_str:
            df.at[idx, datetime_col] = timestamp_str.split('.')[0]
            cleaned_count += 1
    
    if cleaned_count > 0:
        print(f"deleted {cleaned_count} timestamp")
    else:
        print("No microsec found")

    # --- List of formats ---
    date_formats = {
        '1': ('%m/%d/%y %H:%M:%S', 'MM/DD/YY HH:MM:SS'),
        '2': ('%d/%m/%y %H:%M:%S', 'DD/MM/YY HH:MM:SS'),
        '3': ('%m/%d/%Y %H:%M:%S', 'MM/DD/YYYY HH:MM:SS'),
        '4': ('%d/%m/%Y %H:%M:%S', 'DD/MM/YYYY HH:MM:SS'),
        '5': ('%Y-%m-%d %H:%M:%S', 'YYYY-MM-DD HH:MM:SS'),
        '6': ('%Y/%m/%d %H:%M:%S', 'YYYY/MM/DD HH:MM:SS'),
        '7': ('%d-%m-%Y %H:%M:%S', 'DD-MM-YYYY HH:MM:SS'),
        '8': ('%m-%d-%Y %H:%M:%S', 'MM-DD-YYYY HH:MM:SS'),
        '9': ('%m/%d/%y %H:%M', 'MM/DD/YY HH:MM'),
        '10': ('%d/%m/%y %H:%M', 'DD/MM/YY HH:MM'),
        '11': ('%m/%d/%Y %H:%M', 'MM/DD/YYYY HH:MM'),
        '12': ('%d/%m/%Y %H:%M', 'DD/MM/YYYY HH:MM'),
        '13': ('%Y-%d-%m %H:%M:%S', 'YYYY-DD-MM HH:MM:SS'),
        '14': ('%Y/%d/%m %H:%M:%S', 'YYYY/DD/MM HH:MM:SS'),
    }

    # --- delete rows that don't match any format ---
    def try_parse_any_formats(value):
        for code, (fmt, _) in date_formats.items():
            try:
                parsed = datetime.strptime(value, fmt)
                return parsed, code
            except Exception:
                continue
        return None, None

    removed_indices = []
    for idx, val in enumerate(df[datetime_col]):
        parsed, _ = try_parse_any_formats(val)
        if parsed is None:
            removed_indices.append(idx)

    if removed_indices:
        print(f" {len(removed_indices)} deleted rows")
        for idx in removed_indices[:10]:
            print(f"   Row {idx+1}: '{df.at[idx, datetime_col]}'")
        if len(removed_indices) > 10:
            print(f"   ...and other {len(removed_indices)-10} similar rows")
        df = df.drop(index=removed_indices).reset_index(drop=True)
    else:
        print("all lines match at least one format")

    if df.empty:
        print("No valid rows found")
        return df

    # --- Parsing ---
    print("Format list:")
    for key, (fmt, name) in date_formats.items():
        print(f"  {key}. {name}")
    print()

    parsed_dates = []
    problematic_rows = []
    current_format = None
    current_format_code = None
    last_successfully_parsed_value = None

    for idx, value in enumerate(df[datetime_col]):
        original_index = idx + 1
        parsed = None

        if current_format:
            try:
                parsed = pd.to_datetime(value, format=current_format, errors='raise')
                last_successfully_parsed_value = value
            except:
                print(f"\n Row {original_index}: Format change detected!")
                print(f"   Last date parsed: '{last_successfully_parsed_value}'")
                print(f"   Format used: {date_formats[current_format_code][1]}")
                print(f"   New date: '{value}'")
                print(f"\n   This format doesn't work for this date.")
                print(f"   Format list:")
                for key, (fmt, name) in date_formats.items():
                    print(f"     {key}. {name}")

                while True:
                    new_format_choice = input(f"\n   Select the new format (1-{len(date_formats)}): ").strip()

                    if new_format_choice in date_formats:
                        new_format, new_format_name = date_formats[new_format_choice]

                        try:
                            parsed = pd.to_datetime(value, format=new_format, errors='raise')
                            current_format = new_format
                            current_format_code = new_format_choice
                            print(f"   New format set: {new_format_name}")
                            print(f"   Parsed date: {parsed.strftime('%d/%m/%Y %H:%M:%S')}\n")
                            last_successfully_parsed_value = value
                            break
                        except:
                            print(f"   The selected format is not valid for '{value}'. Try again.")
                    else:
                        print(f"   Choose a number between 1 and {len(date_formats)}")
        else:
            print(f"First date detected: '{value}'")
            print(f"   Format list:")
            for key, (fmt, name) in date_formats.items():
                print(f"     {key}. {name}")

            while True:
                format_choice = input(f"\n   Select the format (1-{len(date_formats)}): ").strip()

                if format_choice in date_formats:
                    selected_format, format_name = date_formats[format_choice]

                    try:
                        parsed = pd.to_datetime(value, format=selected_format, errors='raise')
                        current_format = selected_format
                        current_format_code = format_choice
                        print(f"   Format set: {format_name}")
                        print(f"   Parsed data: {parsed.strftime('%d/%m/%Y %H:%M:%S')}\n")
                        last_successfully_parsed_value = value
                        break
                    except:
                        print(f"   The selected format is not valid for '{value}'. Try again.")
                else:
                    print(f"   Choose a number between 1 and {len(date_formats)}")

        if parsed is None:
            parsed_dates.append(None)
            problematic_rows.append((idx, value))
        else:
            parsed_dates.append(parsed)

    df[datetime_col] = parsed_dates

    # --- Handling problematic rows ---
    if len(problematic_rows) > 0:
        print(f"\nFounded {len(problematic_rows)} rows with unrecognizable datetime values:\n")

        for idx, invalid_value in problematic_rows:
            original_index = idx + 1

            print(f"Row {original_index}: '{invalid_value}'")

            while True:
                correction = input(f"  Enter the correct date or press ENTER to delete it: ").strip()

                if correction == "":
                    print(f"  Row {original_index} will be deleted.\n")
                    break
                else:
                    corrected_date = None

                    if current_format:
                        try:
                            corrected_date = pd.to_datetime(correction, format=current_format, errors='raise')
                        except:
                            pass

                    if corrected_date is None:
                        for key, (fmt, name) in date_formats.items():
                            try:
                                corrected_date = pd.to_datetime(correction, format=fmt, errors='raise')
                                break
                            except:
                                continue

                    if corrected_date is not None:
                        df.loc[idx, datetime_col] = corrected_date
                        print(f"  Row {original_index}  corrected with '{corrected_date.strftime('%d/%m/%Y %H:%M:%S')}'.\n")
                        break
                    else:
                        print(f"  Unrecognized format.")

    # Remove rows where the datetime conversion failed
    df = df.dropna(subset=[datetime_col]).reset_index(drop=True)

    if df.empty:
        print("No valid rows found after the datetime cleanup")
        return df

    print(f"\n✓ Dates entered correctly {len(df)} righe")
    print(f"First date: {df[datetime_col].min().strftime('%d/%m/%Y %H:%M:%S')}")
    print(f"Last date: {df[datetime_col].max().strftime('%d/%m/%Y %H:%M:%S')}")

    # Sort by datetime
    df = df.sort_values(datetime_col).reset_index(drop=True)

    # Convert all in DD/MM/YYYY HH:MM:SS format
    df[datetime_col] = df[datetime_col].dt.strftime('%d/%m/%Y %H:%M:%S')

    removed_count = original_len - len(df)
    print(f"\nTotal rows deleted: {removed_count} on {original_len}")
    print(f"Final dataset: {len(df)} rows")
    print(f"All date converted in: DD/MM/YYYY HH:MM:SS\n")

    return df

In [ ]:
filepath = 'change with your path'
percorso = f'{filepath}/MEDA_A_*.xlsx'
# percorso = 'C:/Users/anast/Desktop/SZN/DATI/paper dataset/DATI MEDA/MEDA_B_*.xlsx'
files = glob.glob(percorso)

for file in files:
    nome = os.path.basename(file)
    print(f"\n========== FILE: {nome} ==========")

    df = pd.read_excel(file)
    df_uniformato = uniforma_date(df, datetime_col='Datetime')
    df_uniformato.to_excel(f'{filepath}/{nome}_date_uniformate.xlsx', index=False)

## Invalid values 

In [ ]:
percorso = f'{filepath}/*.xlsx'
files = glob.glob(percorso)

for file in files:
    nome = os.path.basename(file)
    print(f"\n========== FILE: {nome} ==========")
    df = pd.read_excel(file)

    for col in df.columns:
        # Skip the “Datetime” column
        if col == 'Datetime':
            continue
            
        for idx, val in df[col].items():
            if pd.isna(val):
                continue
            try:
                float(val)
            except:
                print(f"[NUM] Row {idx} | COL '{col}' | Value not valid → {val}")
                df.at[idx, col] = np.nan
    
    nuovo_nome = nome.replace(".xlsx", "_cleaned.xlsx")
    output_path = os.path.join(os.path.dirname(file), nuovo_nome)
    df.to_excel(output_path, index=False)
    print(f"File saved: {nuovo_nome}")

In [ ]:
percorso = f'{filepath}/*.xlsx'
cartella_output = 'change with your path'
os.makedirs(cartella_output, exist_ok=True)

nome_file = glob.glob(percorso)

for file in nome_file:
    try:
        df = pd.read_excel(file)
        
        # Replace -32768.0 with NaN
        df = df.replace(-32768.0, pd.NA)
        
        # Create the new file name
        nome_base = os.path.basename(file)
        nome_nuovo = nome_base.replace('.xlsx', '_processed.xlsx')
        percorso_nuovo = os.path.join(cartella_output, nome_nuovo)
        
        df.to_excel(percorso_nuovo, index=False)
        
        print(f"Done: {nome_base}")
        
    except Exception as e:
        print(f"Error with {os.path.basename(file)}: {str(e)}")

print(f"\nFile saved in: {cartella_output}")

## Duplicated test

### Check for duplicate rows

In [ ]:
percorso = f'{filepath}/*.xlsx'
files = glob.glob(percorso)

for file in files:
    nome = os.path.basename(file)
    print(f"\n File: {nome}")

    df = pd.read_excel(file)

    # =========================
    # FIND DUPLICATE ROWS
    # =========================
    dup = df[df.duplicated(keep=False)]

    if dup.empty:
        print(f"No duplicate rows found in {nome}")
        continue

    print(f"Founded {dup.shape[0]} duplicate rows")

    # Print indexes of duplicate rows
    for _, group in dup.groupby(list(df.columns)):
        if len(group) > 1:
            print(f"   Duplicate rows in : {group.index.tolist()}")

    # =========================
    # CREATE A FILE WITHOUT DUPLICATES
    # =========================
    df_nodup = df.drop_duplicates()

    nuovo_nome = nome.replace(".xlsx", "_no_Rdup.xlsx")
    output_path = os.path.join(os.path.dirname(file), nuovo_nome)
    df_nodup.to_excel(output_path, index=False)

    print(f"File without duplicates created: {nuovo_nome}")


### Check for duplicate columns

In [ ]:
percorso = f'{filepath}/*.xlsx'
files = glob.glob(percorso)

for file in files:
    nome = os.path.basename(file)
    print(f"\nFile: {nome}")

    df = pd.read_excel(file)

    # =========================
    # FIND DUPLICATE COLUMNS
    # =========================
    dup_cols = df.T.duplicated(keep=False)

    if not dup_cols.any():
        print("No duplicate columns found")
        continue

    colonne_duplicate = df.columns[dup_cols].tolist()
    print(f"Columns duplicate founded: {colonne_duplicate}")

    for _, group in df.T[dup_cols].groupby(list(df.index)):
        if len(group) > 1:
            print(f"   Duplicate columns: {group.index.tolist()}")

    # =========================
    # CREATE FILE WITHOUT COLUMNS DUPLICATE
    # =========================
    df_nodup = df.loc[:, ~df.T.duplicated()]

    nuovo_nome = nome.replace(".xlsx", "_no_Cdup.xlsx")
    output_path = os.path.join(os.path.dirname(file), nuovo_nome)
    df_nodup.to_excel(output_path, index=False)

    print(f"File without duplicate columns created: {nuovo_nome}")


## Frozen profile test

verify whether each profile has exactly the same values; the fixed threshold is based on the work of Sabia et al. 2019 (it is considered that at least half of the values in the dataset must be equal)

### verify current profiles 

#### velocity

In [ ]:
percorso = 'change with your path'
# Crea la lista dei file
nome_file = glob.glob(percorso)

for file in nome_file:
    df1 = pd.read_excel(file)
    vel_cols = [col for col in df1.columns if "vel (m/sec)" in str(col)]

    # Iterate over each row
    for index, row in df1[vel_cols].iterrows():
        counts = row.value_counts()
        duplicated_values = counts[counts >  (int(len(vel_cols)/2))]
    
        if not duplicated_values.empty:
            print(f"Riga {index}: {duplicated_values.sum()} duplicated values")
            for value, count in duplicated_values.items():
                print(f"  Duplicated value: {value} (repeated {count} times)")

    if file_superato:
        print(f"The file {file} passed the test")

#### direction

In [ ]:
percorso = 'change with your path'
# Crea la lista dei file
nome_file = glob.glob(percorso)

for file in nome_file:
    df1 = pd.read_excel(file)
    dir_cols = [col for col in df1.columns if "dir (deg)" in str(col)]

    file_superato = True

    for index, row in df1[dir_cols].iterrows():
        counts = row.value_counts()
        duplicated_values = counts[counts > int(len(dir_cols) / 2)]

        if not duplicated_values.empty:
            file_superato = False
            print(f"File {file} - Row {index}: {duplicated_values.sum()} duplicated values")
            for value, count in duplicated_values.items():
                print(f"  Duplicated value: {value} (repeated {count} times)")

    # Messaggio finale
    if file_superato:
        print(f"The file {file} passed the test")


### verify each column

In [ ]:
percorso = f'{filepath}/*.xlsx'
# Crea la lista dei file
nome_file = glob.glob(percorso)

for file in nome_file:
    df1 = pd.read_excel(file)
    nome = os.path.basename(file).upper()
    print(f"File: {os.path.basename(file)}")
    
    # -------------------------
    # Current file
    # -------------------------
    if "CORRENTE" in nome: # varies depending on the filename
        cols_da_valutare = [
            col for col in df1.columns
            if ("vel (m/sec)" in str(col).lower()) 
            or ("dir (deg)" in str(col).lower())
            or ("Water level (m)" in str(col))
            or ("Water T" in str(col))
        ]
    # -------------------------
    # Files: WAVE / CTD / METEO
    # -------------------------
    elif any(k in nome for k in ["ONDE", "CTD", "METEO"]): # varies depending on the filename
        cols_da_valutare = [
            col for col in df1.columns
            if str(col).lower() != "datetime"
        ]

    else:
        print("Unrecognized file type → skip")
        continue
    
    if not cols_da_valutare:
        print("No valid columns found")
        continue
    
    for col in cols_da_valutare:
        counts = df1[col].value_counts(dropna=True)
        # Find values that appear in more than half of the rows
        duplicated_values = counts[
            counts > int(len(df1) / 2)
        ]
        
        if not duplicated_values.empty:
            print(f"The column {col} did not pass the test")
            for value, count in duplicated_values.items():
                print(f"  The column {col} has {count} values repeated ({value})")

## Range Test 

In [ ]:
ranges = {

    # ===== Current =====
    'water level (m)': (0, 25),  
    'water T': (-5, 45),
    'vel (m/sec)': (-20, 20), 
    'dir (deg)': (0, 359), 

    # ===== Wave =====
    'Hs (m)': (0, 15),
    'Tp (sec)': (0, 30),
    'Dp (Deg)': (0, 359), 

    'Hs Sea (m)': (0, 20),
    'Tp Sea (sec)': (0, 30),
    'Dp Sea (Deg)': (0, 359), 

    'Hs Swell (m)': (0, 20),
    'Tp Swell (sec)': (0, 30),
    'Dp Swell (Deg)': (0, 359), 

    'Hmax (m)': (0, 30),
    'Tmax (sec)': (0, 30),

    'H 1/3 (m)': (0, 20),
    'T 1/3 (sec)': (0, 30),

    'H 1/10 (m)': (0, 30),
    'T 1/10 (sec)': (0, 30),

    'Hmean (m)': (0, 20),
    'Tmean (sec)': (0, 30),

    'Depth (mm)': (0, 200000), # is equivalent to 200 m

    # ===== CTD =====
    'Temp (°C)': (-5, 35), 
    'Cond (S/m)': (0, 9), 
    'Depth (m)': (0, 350),
    'Sal (PSU)': (12, 42),
    'DO (ml/l)': (2, 9),
    'Oxsat (%)': (0, 120), 
    'Dens (Kg/m3)': (20, 30), # sigma t 
    'Pres (dbar)': (0, 7000), 

    # ===== METEO =====
    'WS (m/s)': (0.1, 60), 
    'WG (m/s)': (0, 80),
    'WD (Deg)': (0, 359),
    'TWD (Deg)': (0, 359), 
    'AT (°C)': (-40, 70),
    'AP (hPa)': (300, 1100) 
}

In [ ]:
percorso = f'{filepath}/*.xlsx'
output_dir = 'change with your path'

os.makedirs(output_dir, exist_ok=True)
files = glob.glob(percorso)
print(f"Founded {len(files)} file to be processed\n")

for file in files:
    base_file = os.path.basename(file)
    
    # Skip temporary files  (they started with ~$)
    if base_file.startswith('~$'):
        print(f"  Skipping temporary file: {base_file}\n")
        continue
    
    print(f'Processing: {base_file}')
    
    try:
        ext = os.path.splitext(file)[1].lower()
        if ext == '.xlsx':
            df = pd.read_excel(file, engine='openpyxl')
        elif ext == '.xls':
            df = pd.read_excel(file, engine='xlrd')
        elif ext == '.xlsm':
            df = pd.read_excel(file, engine='openpyxl')
        else:
            print(f"  ✗ Formato non supportato: {ext}, skipping...\n")
            continue
    except Exception as e:
        print(f"  ✗ Errore nella lettura del file: {e}, skipping...\n")
        continue
    
    print(f"  Original dimensions: {df.shape}")
    
    # STEP 1: Check Depth and Pres
    righe_da_invalidare = pd.Series([False] * len(df), index=df.index)
    colonne_critiche = ['Depth (m)', 'Depth (mm)', 'Pres (dbar)']
    
    for col in colonne_critiche:
        if col in df.columns and col in ranges:
            vmin, vmax = ranges[col]
            mask = (df[col] < vmin) | (df[col] > vmax)
            n_fuori_range = mask.sum()
            
            if n_fuori_range > 0:
                indici_fuori = df.index[mask].tolist()
                print(f"  ! {col}: {n_fuori_range} values outside the range [{vmin}, {vmax}]")
                print(f"    Indices: {indici_fuori[:10]}{'...' if len(indici_fuori) > 10 else ''}")
                righe_da_invalidare |= mask
    
    # Invalidate entire rows where Depth/Pres are out of range
    n_righe_invalidate = righe_da_invalidare.sum()
    if n_righe_invalidate > 0:
        indici_invalidati = df.index[righe_da_invalidare].tolist()
        print(f"  → {n_righe_invalidate} entire lines invalidated")
        print(f"    Indici righe invalidate: {indici_invalidati[:10]}{'...' if len(indici_invalidati) > 10 else ''}")
        df.loc[righe_da_invalidare, :] = np.nan
    
    # STEP 2: Apply the range test to ALL columns
    valori_modificati = 0
    
    for col in df.columns:
        range_trovato = False
        vmin = vmax = None
        
        # Ccheck for columns
        if col in ranges:
            vmin, vmax = ranges[col]
            range_trovato = True
        else:
            # Check for current columns (es: "1 vel (m/s)", "2 dir (deg)", ecc.)
            for key in ranges.keys():
                if key in col:
                    vmin, vmax = ranges[key]
                    range_trovato = True
                    break
        
        if range_trovato:
            mask = (df[col] < vmin) | (df[col] > vmax)
            n_modificati = mask.sum()
            
            if n_modificati > 0:
                indici_modificati = df.index[mask].tolist()
                print(f"  - {col}: {n_modificati} valori fuori range [{vmin}, {vmax}] → NaN")
                print(f"    Indici: {indici_modificati[:10]}{'...' if len(indici_modificati) > 10 else ''}")
                valori_modificati += n_modificati
            
            df.loc[mask, col] = np.nan
    
    # STEP 3: Save the file
    base_name = os.path.splitext(base_file)[0]
    new_name = base_name + "_RT.xlsx"
    output_path = os.path.join(output_dir, new_name)
    
    df.to_excel(output_path, index=False, engine='openpyxl')
    

    if os.path.exists(output_path):
        size = os.path.getsize(output_path)
        print(f"  File saved: {new_name} ({size/1024:.2f} KB)")
    else:
        print(f"  ERROR: File not saved!")
    
    print(f"  Total modified values: {valori_modificati}\n")

## Spike test

## check outlier using a moving window
After running the range test, check for outliers in all parameters except for the directions

In [ ]:
percorso = f'{filepath}/*.xlsx'
nome_file = glob.glob(percorso)

for file in nome_file:

    df3m = pd.read_excel(file)

    # Check Datetime
    if 'Datetime' not in df3m.columns:
        continue

    df3m['Datetime'] = pd.to_datetime(df3m['Datetime'])
    df3m = df3m.sort_values('Datetime').set_index('Datetime')

    df3m['Month'] = df3m.index.to_period('M')

    # Columns to exclude 
    cols_to_exclude = [
        'Datetime',
        'Week',
        'ensemble number',
        'start bin',
        'last effective bin',
        'step bin',
        'Dp (Deg)',
        'Dp Sea (Deg)',
        'Dp Swell (Deg)',
        'WD (Deg)',
        'TWD (Deg)'
    ]
    cols_to_exclude_lower = [c.lower() for c in cols_to_exclude]

    # Select columns to process (except direction in current file)
    colonne_da_processare = [
        col for col in df3m.columns
        if col.lower() not in cols_to_exclude_lower
        and pd.api.types.is_numeric_dtype(df3m[col])
        and not re.search(r'\bdir\s*\(deg\)', col.lower())
    ]

    df_senza_outlier = df3m.copy()
    df_senza_outlier_estremi = df3m.copy()

    for layer in colonne_da_processare:

        Q1 = pd.Series(index=df3m.index, dtype='float64')
        Q3 = pd.Series(index=df3m.index, dtype='float64')

        for month, idx_month in df3m.groupby('Month').groups.items():

            start_month = month.to_timestamp(how='start')
            end_month = month.to_timestamp(how='end')

            # Extended window: 15 days of the previous month
            start_window = start_month - pd.Timedelta(days=15)

            dati_finestra = df3m.loc[
                (df3m.index >= start_window) &
                (df3m.index <= end_month),
                layer
            ].dropna()

            if len(dati_finestra) == 0:
                continue

            q1 = dati_finestra.quantile(0.25)
            q3 = dati_finestra.quantile(0.75)

            Q1.loc[idx_month] = q1
            Q3.loc[idx_month] = q3

        IQR = Q3 - Q1
        serie = df3m[layer]

        # Dynamic thresholds
        mild_lower = Q1 - 1.5 * IQR
        mild_upper = Q3 + 1.5 * IQR
        extreme_lower = Q1 - 3 * IQR
        extreme_upper = Q3 + 3 * IQR

        # Mask outlier
        is_extreme_outlier = (serie < extreme_lower) | (serie > extreme_upper)
        is_mild_outlier = (
            ((serie < mild_lower) & (serie >= extreme_lower)) |
            ((serie > mild_upper) & (serie <= extreme_upper))
        )

        # Remove outlier
        df_senza_outlier.loc[is_extreme_outlier | is_mild_outlier, layer] = np.nan
        df_senza_outlier_estremi.loc[is_extreme_outlier, layer] = np.nan

    # Remove column Month
    df_senza_outlier = df_senza_outlier.drop('Month', axis=1)
    df_senza_outlier_estremi = df_senza_outlier_estremi.drop('Month', axis=1)

    # Reset index
    df_senza_outlier = df_senza_outlier.reset_index()
    df_senza_outlier_estremi = df_senza_outlier_estremi.reset_index()

    nome = os.path.basename(file).replace(".xlsx", "")

    df_senza_outlier.to_excel(
        f"{filepath}/{nome}_no_outlier_mensile.xlsx",
        index=False
    )

    df_senza_outlier_estremi.to_excel(
        f"{filepath}/{nome}_no_outlier_estremi_mensile.xlsx",
        index=False
    )

## nan dir 

Link the direction data to the speed; if the speed is null, the associated direction must also be null
* current: vel -> dir 
* wave: hs -> dp; hs sea -> dp sea; Hs swell -> dp swell
* meteo: ws -> wd and twd 


In [ ]:
input_path = f'{filepath}/*.xlsx'
output_dir = 'change with your path'
os.makedirs(output_dir, exist_ok=True)

files = glob.glob(input_path)


def applica_nan_coerenti(df):

    # -------- Current --------
    vel_pattern = re.compile(r'^(\d+)\s*vel\s*\(m/sec\)', re.IGNORECASE)
    dir_pattern = re.compile(r'^(\d+)\s*dir\s*\(deg\)', re.IGNORECASE)

    vel_cols = {vel_pattern.match(c).group(1): c
                for c in df.columns if vel_pattern.match(c)}
    dir_cols = {dir_pattern.match(c).group(1): c
                for c in df.columns if dir_pattern.match(c)}

    for k in vel_cols:
        if k in dir_cols:
            df.loc[df[vel_cols[k]].isna(), dir_cols[k]] = np.nan

    # -------- Wave --------
    wave_groups = [
        ('Hs (m)', 'Tp (sec)', 'Dp (Deg)'),
        ('Hs Sea (m)', 'Tp Sea (sec)', 'Dp Sea (Deg)'),
        ('Hs Swell (m)', 'Tp Swell (sec)', 'Dp Swell (Deg)')
    ]

    for hs, tp, dp in wave_groups:
        if hs in df.columns:
            if tp in df.columns:
                df.loc[df[hs].isna(), tp] = np.nan
            if dp in df.columns:
                df.loc[df[hs].isna(), dp] = np.nan

    # -------- METEO --------
    if 'WS (m/s)' in df.columns:
        for col in ['WD (Deg)', 'TWD (Deg)']:
            if col in df.columns:
                df.loc[df['WS (m/s)'].isna(), col] = np.nan

    return df


for file in files:

    df = pd.read_excel(file)

    df = applica_nan_coerenti(df)

    nome = os.path.basename(file)
    out_file = os.path.join(output_dir, nome.replace('.xlsx', '_coerente_nan.xlsx'))

    df.to_excel(out_file, index=False)

    print(f'Create: {out_file}')

## Missing values

In [ ]:
def detect_and_fill_gaps(df, datetime_col='Datetime', expected_interval_minutes=15, tolerance_minutes=5):
    """
    Detects time gaps in the dataset and fills them with NaN rows up to December 31
    of the year of the latest timestamp in the DataFrame.

    Parameters:
    - df: DataFrame containing the data
    - datetime_col: name of the datetime column
    - expected_interval_minutes: expected interval between measurements (default 15 min)
    - tolerance_minutes: tolerance for considering an interval valid (default 5 min)
    """

    original_len = len(df)
    df = df.dropna(subset=[datetime_col]).reset_index(drop=True)
    df[datetime_col] = df[datetime_col].astype(str)

    # --- delete microsec/nanosec  ---
    cleaned_count = 0
    for idx in range(len(df)):
        timestamp_str = df.at[idx, datetime_col]
        if '.' in timestamp_str:
            df.at[idx, datetime_col] = timestamp_str.split('.')[0]
            cleaned_count += 1
    
    if cleaned_count > 0:
        print(f"deleted {cleaned_count} timestamp")
    else:
        print("No microsec found")

    # --- Format list ---
    date_formats = {
        '1': ('%m/%d/%y %H:%M:%S', 'MM/DD/YY HH:MM:SS'),
        '2': ('%d/%m/%y %H:%M:%S', 'DD/MM/YY HH:MM:SS'),
        '3': ('%m/%d/%Y %H:%M:%S', 'MM/DD/YYYY HH:MM:SS'),
        '4': ('%d/%m/%Y %H:%M:%S', 'DD/MM/YYYY HH:MM:SS'),
        '5': ('%Y-%m-%d %H:%M:%S', 'YYYY-MM-DD HH:MM:SS'),
        '6': ('%Y/%m/%d %H:%M:%S', 'YYYY/MM/DD HH:MM:SS'),
        '7': ('%d-%m-%Y %H:%M:%S', 'DD-MM-YYYY HH:MM:SS'),
        '8': ('%m-%d-%Y %H:%M:%S', 'MM-DD-YYYY HH:MM:SS'),
        '9': ('%m/%d/%y %H:%M', 'MM/DD/YY HH:MM'),
        '10': ('%d/%m/%y %H:%M', 'DD/MM/YY HH:MM'),
        '11': ('%m/%d/%Y %H:%M', 'MM/DD/YYYY HH:MM'),
        '12': ('%d/%m/%Y %H:%M', 'DD/MM/YYYY HH:MM'),
        '13': ('%Y-%d-%m %H:%M:%S', 'YYYY-DD-MM HH:MM:SS'),
        '14': ('%Y/%d/%m %H:%M:%S', 'YYYY/DD/MM HH:MM:SS'),
    }

    # --- delete rows that don't match any format---
    def try_parse_any_formats(value):
        for code, (fmt, _) in date_formats.items():
            try:
                parsed = datetime.strptime(value, fmt)
                return parsed, code
            except Exception:
                continue
        return None, None

    removed_indices = []
    for idx, val in enumerate(df[datetime_col]):
        parsed, _ = try_parse_any_formats(val)
        if parsed is None:
            removed_indices.append(idx)

    if removed_indices:
        print(f"{len(removed_indices)} deleted rows")
        for idx in removed_indices[:10]:
            print(f"   Row {idx+1}: '{df.at[idx, datetime_col]}'")
        if len(removed_indices) > 10:
            print(f"   ...and other {len(removed_indices)-10} similar rows")
        df = df.drop(index=removed_indices).reset_index(drop=True)
    else:
        print("all lines match at least one format")

    if df.empty:
        print("No valid lines found")
        return df

    # --- Parsing ---
    print("Format list:")
    for key, (fmt, name) in date_formats.items():
        print(f"  {key}. {name}")
    print()

    parsed_dates = []
    problematic_rows = []
    current_format = None
    current_format_code = None
    last_successfully_parsed_value = None

    for idx, value in enumerate(df[datetime_col]):
        original_index = idx + 1
        parsed = None

        if current_format:
            try:
                parsed = pd.to_datetime(value, format=current_format, errors='raise')
                last_successfully_parsed_value = value
            except:
                print(f"\n Row {original_index}: Format change detected!")
                print(f"   Last date parsed: '{last_successfully_parsed_value}'")
                print(f"   Format used: {date_formats[current_format_code][1]}")
                print(f"   New date: '{value}'")
                print(f"\n   This format doesn't work for this date.")
                print(f"   Format list:")
                for key, (fmt, name) in date_formats.items():
                    print(f"     {key}. {name}")

                while True:
                    new_format_choice = input(f"\n   Select the new format (1-{len(date_formats)}): ").strip()

                    if new_format_choice in date_formats:
                        new_format, new_format_name = date_formats[new_format_choice]
                        try:
                            parsed = pd.to_datetime(value, format=new_format, errors='raise')
                            current_format = new_format
                            current_format_code = new_format_choice
                            print(f"   New format set: {new_format_name}")
                            print(f"   Parsed date: {parsed.strftime('%d/%m/%Y %H:%M:%S')}\n")
                            last_successfully_parsed_value = value
                            break
                        except:
                            print(f"   The selected format is not valid for '{value}'. Try again.")
                    else:
                        print(f"   Choose a number between 1 and {len(date_formats)}")
        else:
            print(f"First date detected: '{value}'")
            print(f"   Format list:")
            for key, (fmt, name) in date_formats.items():
                print(f"     {key}. {name}")

            while True:
                format_choice = input(f"\n   Select the format (1-{len(date_formats)}): ").strip()

                if format_choice in date_formats:
                    selected_format, format_name = date_formats[format_choice]
                    try:
                        parsed = pd.to_datetime(value, format=selected_format, errors='raise')
                        current_format = selected_format
                        current_format_code = format_choice
                        print(f"   Format set: {format_name}")
                        print(f"   Date parsed: {parsed.strftime('%d/%m/%Y %H:%M:%S')}\n")
                        last_successfully_parsed_value = value
                        break
                    except:
                        print(f"   The selected format is not valid '{value}'. Try again.")
                else:
                    print(f"   Choose a number between 1 and {len(date_formats)}")

        if parsed is None:
            parsed_dates.append(None)
            problematic_rows.append((idx, value))
        else:
            parsed_dates.append(parsed)

    # Assegna le date parsate
    df[datetime_col] = parsed_dates

    # Gestisci le righe problematiche
    if len(problematic_rows) > 0:
        print(f"\nFound {len(problematic_rows)} rows with unrecognizable datetimes:\n")

        for idx, invalid_value in problematic_rows:
            original_index = idx + 1
            print(f"Row {original_index}: '{invalid_value}'")

            while True:
                correction = input(f"  Enter the correct date or press ENTER to delete it: ").strip()

                if correction == "":
                    print(f"  Row {original_index} will be deleted.\n")
                    break
                else:
                    corrected_date = None

                    if current_format:
                        try:
                            corrected_date = pd.to_datetime(correction, format=current_format, errors='raise')
                        except:
                            pass

                    if corrected_date is None:
                        for key, (fmt, name) in date_formats.items():
                            try:
                                corrected_date = pd.to_datetime(correction, format=fmt, errors='raise')
                                break
                            except:
                                continue

                    if corrected_date is not None:
                        df.loc[idx, datetime_col] = corrected_date
                        print(f"  Row {original_index} correct with '{corrected_date.strftime('%d/%m/%Y %H:%M:%S')}'.\n")
                        break
                    else:
                        print(f"  Unrecognized format.")

    # Rimuovi righe dove la conversione datetime è fallita
    df = df.dropna(subset=[datetime_col]).reset_index(drop=True)

    if df.empty:
        print("No valid rows found after datetime cleanup!")
        return df

    print(f"\nDates parsed correctly: {len(df)} righe")
    print(f"First date: {df[datetime_col].min().strftime('%d/%m/%Y %H:%M:%S')}")
    print(f"Last date: {df[datetime_col].max().strftime('%d/%m/%Y %H:%M:%S')}")

    # Dort for datetime
    df = df.sort_values(datetime_col).reset_index(drop=True)

    # Calculate time differences
    time_diffs = df[datetime_col].diff()
    max_expected_diff = timedelta(minutes=expected_interval_minutes + tolerance_minutes)
    gaps = time_diffs > max_expected_diff

    all_rows = []
    for i, row in df.iterrows():
        all_rows.append(row)
        if i < len(df) - 1 and gaps.iloc[i + 1]:
            current_time = row[datetime_col]
            next_time = df.iloc[i + 1][datetime_col]
            time_diff = next_time - current_time
            missing_measurements = int(time_diff.total_seconds() / (expected_interval_minutes * 60)) - 1
            for j in range(1, missing_measurements + 1):
                gap_time = current_time + timedelta(minutes=expected_interval_minutes * j)
                gap_row = row.copy()
                gap_row[datetime_col] = gap_time
                for col in df.columns:
                    if col != datetime_col:
                        gap_row[col] = np.nan
                all_rows.append(gap_row)

    result_df = pd.DataFrame(all_rows).reset_index(drop=True)

    result_df[datetime_col] = pd.to_datetime(result_df[datetime_col], errors='coerce')

    result_df = result_df.sort_values(datetime_col).reset_index(drop=True)

    result_df[datetime_col] = pd.to_datetime(result_df[datetime_col], errors='coerce')

    result_df = result_df.sort_values(datetime_col).reset_index(drop=True)

    # Convert in DD/MM/YYYY HH:MM:SS format
    result_df[datetime_col] = result_df[datetime_col].dt.strftime('%d/%m/%Y %H:%M:%S')

    final_removed = original_len - len(result_df)

    print(f"Final dataset: {len(result_df)} rows with intervals of {expected_interval_minutes} minutes")
    print(f"Total deleted rows: {final_removed} su {original_len}")

    return result_df

In [ ]:
percorso = f'{filepath}/*.xlsx' 
nome_file = glob.glob(percorso)

for file in nome_file:
    df1 = pd.read_excel(file)
    df_with_gaps_filled = detect_and_fill_gaps(
        df1, 
        datetime_col='Datetime', 
        expected_interval_minutes=60, 
        tolerance_minutes=5
    )
    nome = os.path.basename(file).replace(".xlsx", "")

    df_with_gaps_filled.to_excel(f'{filepath}/{nome}_added_missing_val.xlsx', index=False) 